# 第01章 B1组独立验证

**小组 G15／B1。** A1 与 A2 的被审版本均为 `fa7f1f232017ae827bd82d8280de8b6e2fdc4c83`。

本Notebook使用Python标准库对两份A稿四节主线作独立数值对照。float32值通过标准库`struct`按IEEE binary32舍入。参考关系、断言和输出按节给出。推导和代码在阅稿后形成，详见[独立推导](B1组-独立推导.md)。

## 运行环境与人工核查

**已人工运行核查。** 2026-11-09，G15组员在本机 JupyterLab 运行本Notebook，反馈“没有红色报错”。本文件共有五个代码格。

保存在各格中的具体数值来自新建的 Linux Python 3.12.14 虚拟环境。G15本机运行的证据见[验证报告](B1组-验证报告.md)。

In [1]:
import math, platform, struct, sys
from decimal import Decimal, localcontext

def f32(value):
    """将数值按IEEE binary32存储后再转回Python float。"""
    return struct.unpack("!f", struct.pack("!f", float(value)))[0]

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Dependencies: Python standard library only")
print("Device: CPU; random sampling: none")


Python: 3.12.14
Platform: Linux-6.18.57-x86_64-with-glibc2.39
Dependencies: Python standard library only
Device: CPU; random sampling: none


## 1.1 浮点数与舍入误差

检查两种精度的`1`处间距、十进制0.1+0.2，以及相同数值换顺序后的求和。这里的判据是精确格式关系和已知准确和，不使用模糊的显示位数。

In [2]:
spacing = {
    "float32": {"p": 24, "gap_at_1": 2.0 ** -23, "unit_roundoff": 2.0 ** -24},
    "float64": {"p": 53, "gap_at_1": math.nextafter(1.0, 2.0) - 1.0,
                "unit_roundoff": (math.nextafter(1.0, 2.0) - 1.0) / 2},
}
assert f32(1.0 + spacing["float32"]["gap_at_1"]) > 1.0
assert spacing["float64"]["gap_at_1"] == 2.0 ** -52

sum_rows = {}
for name, cast in (("float32", f32), ("float64", float)):
    small = cast(cast(0.1) + cast(0.2))
    order_a = cast(cast(cast(1e16) + cast(1)) + cast(-1e16))
    order_b = cast(cast(cast(1e16) + cast(-1e16)) + cast(1))
    assert Decimal.from_float(float(small)) != Decimal("0.3")
    assert float(order_a) == 0.0 and float(order_b) == 1.0
    sum_rows[name] = {
        "0.1_plus_0.2": float(small),
        "equals_rounded_float_0.3": bool(small == cast(0.3)),
        "exactly_equals_decimal_0.3": Decimal.from_float(float(small)) == Decimal("0.3"),
        "order_1e16_plus_1_minus_1e16": float(order_a),
        "order_1e16_minus_1e16_plus_1": float(order_b),
    }
print("spacing:", spacing)
print("sum checks:", sum_rows)


spacing: {'float32': {'p': 24, 'gap_at_1': 1.1920928955078125e-07, 'unit_roundoff': 5.960464477539063e-08}, 'float64': {'p': 53, 'gap_at_1': 2.220446049250313e-16, 'unit_roundoff': 1.1102230246251565e-16}}
sum checks: {'float32': {'0.1_plus_0.2': 0.30000001192092896, 'equals_rounded_float_0.3': True, 'exactly_equals_decimal_0.3': False, 'order_1e16_plus_1_minus_1e16': 0.0, 'order_1e16_minus_1e16_plus_1': 1.0}, 'float64': {'0.1_plus_0.2': 0.30000000000000004, 'equals_rounded_float_0.3': False, 'exactly_equals_decimal_0.3': False, 'order_1e16_plus_1_minus_1e16': 0.0, 'order_1e16_minus_1e16_plus_1': 1.0}}


## 1.2 截断误差与灾难性抵消

用代数有理化作为独立算法对照，并扫描`h=10^-1,...,10^-16`的指数函数前向差分。小步长扫描的最小误差仅描述本次binary64实现。

In [3]:
x = 1e16
direct = math.sqrt(x + 1.0) - math.sqrt(x)
rationalized = 1.0 / (math.sqrt(x + 1.0) + math.sqrt(x))
with localcontext() as ctx:
    ctx.prec = 80
    xd = Decimal("1e16")
    reference = Decimal(1) / ((xd + 1).sqrt() + xd.sqrt())
    rationalized_error = abs(Decimal.from_float(rationalized) - reference) / reference
assert direct == 0.0
assert rationalized > 0.0 and rationalized_error < Decimal("1e-15")
print("sqrt difference:", {"direct": direct, "rationalized": rationalized,
      "80-digit reference": str(reference), "relative error of rationalized": str(rationalized_error)})

fd_rows = []
for k in range(1, 17):
    h = 10.0 ** (-k)
    estimate = (math.exp(1.0 + h) - math.exp(1.0)) / h
    fd_rows.append((h, abs(estimate - math.e), estimate))
best = min(fd_rows, key=lambda row: row[1])
assert fd_rows[-1][2] == 0.0  # 1+h rounds to 1 in binary64 at this scale.
assert best[0] not in (1e-1, 1e-16)
print("finite difference scan:", {"best_scanned_h": best[0], "best_abs_error": best[1],
      "h_1e-16_estimate": fd_rows[-1][2], "h_1e-16_abs_error": fd_rows[-1][1]})


sqrt difference: {'direct': 0.0, 'rationalized': 5e-09, '80-digit reference': '4.9999999999999998750000000000000062499999999999996093750000000000273437500000000E-9', 'relative error of rationalized': '4.5922560830128472573390654842499671161196757352094493741665234464442049186053977E-17'}
finite difference scan: {'best_scanned_h': 1e-08, 'best_abs_error': 6.60275079056305e-09, 'h_1e-16_estimate': 0.0, 'h_1e-16_abs_error': 2.718281828459045}


## 1.3 条件数与稳定性

对`q(x)=1/(1-x)`，直接由导数推导局部相对条件数；用高精度有限扰动关系核对，并再次对照良态平方根差的两种实现。

In [4]:
with localcontext() as ctx:
    ctx.prec = 80
    x = Decimal("0.9999")
    kappa = abs(x / (1 - x))
    rows = []
    for d_text in ("1e-6", "1e-8", "1e-10"):
        d = Decimal(d_text)
        relative_input = abs(d / x)
        q0 = 1 / (1 - x)
        q1 = 1 / (1 - x - d)
        relative_output = abs((q1 - q0) / q0)
        amplification = relative_output / relative_input
        closed_form = x / (1 - x - d)
        assert abs(amplification - closed_form) < Decimal("1e-70")
        rows.append({"d": d_text, "amplification": str(amplification), "closed_form": str(closed_form)})
    assert kappa == Decimal(9999)
print("local relative condition number:", str(kappa))
print("finite perturbations:", rows)
print("interpretation: condition number is about the problem; cancellation in the direct square-root formula is about the algorithm.")


local relative condition number: 9999
finite perturbations: [{'d': '1e-6', 'amplification': '10099.999999999999999999999999999999999999999999999999999999999999999999999999990', 'closed_form': '1.01E+4'}, {'d': '1e-8', 'amplification': '9999.9999999999999999999999999999999999999999999999999999999999999999999999990001', 'closed_form': '1E+4'}, {'d': '1e-10', 'amplification': '9999.0099990099990099990099990099990099990099990099990099990099990099990099990001', 'closed_form': '9999.0099990099990099990099990099990099990099990099990099990099990099990099990100'}]
interpretation: condition number is about the problem; cancellation in the direct square-root formula is about the algorithm.


## 1.4 AI关联：训练中的数值问题

比较直接softmax与减去最大值后的softmax，并在极端类别概率下计算log域损失。最后检查float32中的微小参数更新。

In [5]:
def direct_softmax(z):
    try:
        e = [math.exp(float(v)) for v in z]
    except OverflowError:
        return [float("nan")] * len(z)
    total = sum(e)
    if total == 0.0:
        return [float("nan")] * len(z)
    return [v / total for v in e]

def shifted_softmax(z):
    z = [float(v) for v in z]
    m = max(z)
    e = [math.exp(v - m) for v in z]
    total = sum(e)
    return [v / total for v in e]

softmax_rows = []
for z in ([1000, 1001, 1002], [-1000, -1001, -1002]):
    naive = direct_softmax(z)
    stable = shifted_softmax(z)
    assert not all(math.isfinite(v) for v in naive)
    assert all(math.isfinite(v) for v in stable) and abs(sum(stable) - 1.0) <= 1e-14
    softmax_rows.append({"logits": list(z), "direct": naive, "shifted": stable,
                         "shifted_sum": sum(stable)})

z = [1000.0, 0.0, -1000.0]
m = max(z)
logsumexp = m + math.log(sum(math.exp(v - m) for v in z))
loss_for_last_class = logsumexp - z[-1]
last_probability = shifted_softmax(z)[-1]
assert loss_for_last_class == 2000.0 and last_probability == 0.0
update32 = f32(f32(1.0) - f32(1e-8))
update64 = 1.0 - 1e-8
assert update32 == 1.0 and update64 < 1.0
print("softmax comparisons:", softmax_rows)
print("log-domain loss and probability:", {"loss": loss_for_last_class, "last_probability_binary64": last_probability})
print("small update:", {"float32": update32, "float64": update64})


softmax comparisons: [{'logits': [1000, 1001, 1002], 'direct': [nan, nan, nan], 'shifted': [0.09003057317038046, 0.24472847105479764, 0.6652409557748218], 'shifted_sum': 0.9999999999999999}, {'logits': [-1000, -1001, -1002], 'direct': [nan, nan, nan], 'shifted': [0.6652409557748218, 0.24472847105479764, 0.09003057317038046], 'shifted_sum': 0.9999999999999999}]
log-domain loss and probability: {'loss': 2000.0, 'last_probability_binary64': 0.0}
small update: {'float32': 1.0, 'float64': 0.99999999}


## 验证范围

保存运行中的数值断言均通过。对照范围为上述输入和四节代表性计算。A稿的来源核对、习题逐项审核及修订状态见[验证报告](B1组-验证报告.md)与[修正清单](B1组-修正清单.md)。